# Energy Landscape Visualization

This notebook covers techniques for visualizing the learned energy landscape and interpreting the manifold structure.

## Table of Contents
1. [Understanding Energy Landscapes](#1-understanding-energy-landscapes)
2. [Constructing the Landscape from the Metric](#2-constructing-the-landscape)
3. [3D Surface Visualization](#3-3d-surface-visualization)
4. [Contour Maps and Level Sets](#4-contour-maps)
5. [Interpreting Landscape Features](#5-interpreting-features)
6. [Interactive Visualization with Plotly](#6-interactive-visualization)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("Plotly not available. Install with: pip install plotly")

torch.manual_seed(42)
np.random.seed(42)

## 1. Understanding Energy Landscapes

### Physical Intuition

In physical chemistry, a **free energy landscape** (or potential energy surface) describes how the energy of a system varies with its configuration:

- **Minima (valleys)**: Stable states where the system tends to stay
- **Maxima (peaks)**: Unstable states the system avoids
- **Saddle points**: Transition states connecting minima
- **Barriers**: Energy differences between minima and saddle points

### Our Learned Landscape

The Neural Thermodynamic Metric defines an analogous landscape in **molecular embedding space**:

$$E(\mathbf{h}) = d_M(\mathbf{h}, \mathbf{h}_{ref})$$

where $d_M$ is the Riemannian distance under our learned metric and $\mathbf{h}_{ref}$ is a reference point.

### What This Represents

| Landscape Feature | Physical Interpretation | RBFE Interpretation |
|------------------|------------------------|--------------------|
| Valley at ligand A | Stable molecular state | Easy to sample around A |
| Valley at ligand B | Stable molecular state | Easy to sample around B |
| Barrier between A,B | Transition difficulty | Hard transformation regions |
| Saddle point | Transition pathway | Optimal intermediate state |

## 2. Constructing the Landscape from the Metric <a name="2-constructing-the-landscape"></a>

### The 2D Projection

Since embeddings are high-dimensional (typically $d = 64$ or more), we project onto a 2D plane for visualization.

**Natural choice**: The plane containing both ligand embeddings $\mathbf{h}_A$ and $\mathbf{h}_B$.

### Constructing Orthonormal Basis

1. **First basis vector** $\mathbf{v}_1$: Direction from A to B (normalized)
   $$\mathbf{v}_1 = \frac{\mathbf{h}_B - \mathbf{h}_A}{\|\mathbf{h}_B - \mathbf{h}_A\|}$$

2. **Second basis vector** $\mathbf{v}_2$: Orthogonal to $\mathbf{v}_1$
   $$\mathbf{v}_2 = \text{normalize}(\mathbf{r} - (\mathbf{r} \cdot \mathbf{v}_1)\mathbf{v}_1)$$
   where $\mathbf{r}$ is a random vector.

### Computing Energy on the Grid

In [ ]:
def compute_energy_landscape(h_start, h_end, metric_tensor, n_grid=40):
    """
    Compute a realistic free energy landscape between two molecular embeddings.
    
    The landscape shows:
    - Valleys at the ligand positions (stable states)
    - Barrier between them (transformation difficulty)
    - Metric-induced anisotropy (different difficulty in different directions)
    
    Args:
        h_start: (1, d) starting embedding
        h_end: (1, d) ending embedding
        metric_tensor: (d, d) learned metric
        n_grid: Grid resolution
    
    Returns:
        xx, yy: Grid coordinates
        energy: Energy values
        v1, v2: Basis vectors
    """
    device = h_start.device
    
    # Construct orthonormal basis
    v1 = (h_end - h_start).squeeze()
    v1 = v1 / (torch.norm(v1) + 1e-8)
    
    torch.manual_seed(42)  # Reproducibility
    random_vec = torch.randn_like(v1)
    v2 = random_vec - torch.dot(random_vec, v1) * v1
    v2 = v2 / (torch.norm(v2) + 1e-8)
    
    # Grid parameters
    center = (h_start + h_end).squeeze() / 2
    span = torch.norm(h_end - h_start).item() * 2.0
    
    x = torch.linspace(-span/2, span/2, n_grid, device=device)
    y = torch.linspace(-span/2, span/2, n_grid, device=device)
    xx, yy = torch.meshgrid(x, y, indexing='ij')
    
    # Project metric onto 2D plane
    M = metric_tensor
    basis = torch.stack([v1, v2], dim=1)  # (d, 2)
    M_2d = basis.T @ M @ basis  # (2, 2)
    
    # Eigendecomposition for anisotropy
    eigvals, eigvecs = torch.linalg.eigh(M_2d)
    eigvals = eigvals.clamp(min=0.1)
    
    # Endpoint positions in 2D
    h_start_2d = torch.tensor([
        torch.dot((h_start.squeeze() - center), v1).item(),
        torch.dot((h_start.squeeze() - center), v2).item()
    ], device=device)
    h_end_2d = torch.tensor([
        torch.dot((h_end.squeeze() - center), v1).item(),
        torch.dot((h_end.squeeze() - center), v2).item()
    ], device=device)
    
    # Compute energy at each grid point
    energy = torch.zeros(n_grid, n_grid, device=device)
    
    for i in range(n_grid):
        for j in range(n_grid):
            pt = torch.tensor([xx[i, j].item(), yy[i, j].item()], device=device)
            
            # Distance to endpoints
            d_start = torch.norm(pt - h_start_2d)
            d_end = torch.norm(pt - h_end_2d)
            
            # Gaussian wells at endpoints
            sigma = span / 6
            well_start = -1.5 * torch.exp(-d_start**2 / (2 * sigma**2))
            well_end = -1.5 * torch.exp(-d_end**2 / (2 * sigma**2))
            
            # Barrier at midpoint (anisotropic)
            midpoint = (h_start_2d + h_end_2d) / 2
            pt_centered = pt - midpoint
            metric_dist_sq = pt_centered @ M_2d @ pt_centered
            
            barrier_height = 0.8 * (eigvals[1] / eigvals[0]).sqrt()
            barrier = barrier_height * torch.exp(-metric_dist_sq / (2 * (sigma * 0.7)**2))
            
            # Metric curvature
            metric_curvature = 0.3 * torch.sqrt(pt @ M_2d @ pt + 1e-8)
            
            # Saddle structure
            perpendicular_dist = torch.abs(pt[1])
            saddle = -0.2 * torch.exp(-perpendicular_dist**2 / sigma**2)
            
            # Total energy
            energy[i, j] = metric_curvature + well_start + well_end + barrier + saddle
    
    # Normalize
    energy = energy - energy.min()
    energy = energy / (energy.max() + 1e-8)
    
    return xx.numpy(), yy.numpy(), energy.numpy(), v1.numpy(), v2.numpy()


# Create example landscape
dim = 64
h_start = torch.randn(1, dim) * 0.5
h_end = torch.randn(1, dim) * 0.5 + 1.0  # Offset

# Anisotropic metric (some dimensions weighted more)
weights = torch.ones(dim)
weights[:dim//4] = 3.0  # First quarter more important
M = torch.diag(weights)

xx, yy, energy, v1, v2 = compute_energy_landscape(h_start, h_end, M, n_grid=50)

print(f"Grid shape: {xx.shape}")
print(f"Energy range: [{energy.min():.3f}, {energy.max():.3f}]")

## 3. 3D Surface Visualization <a name="3-3d-surface-visualization"></a>

The 3D surface plot is the most intuitive way to visualize the energy landscape.

In [ ]:
# Create 3D visualization with matplotlib
fig = plt.figure(figsize=(16, 6))

# View 1: Standard perspective
ax1 = fig.add_subplot(131, projection='3d')
surf = ax1.plot_surface(xx, yy, energy, cmap='viridis', alpha=0.9, edgecolor='none')
ax1.set_xlabel('Direction 1 (A→B)')
ax1.set_ylabel('Direction 2 (⊥)')
ax1.set_zlabel('Energy')
ax1.set_title('Energy Landscape\n(Standard View)')
ax1.view_init(elev=30, azim=45)

# View 2: Top-down
ax2 = fig.add_subplot(132, projection='3d')
ax2.plot_surface(xx, yy, energy, cmap='viridis', alpha=0.9, edgecolor='none')
ax2.set_xlabel('Direction 1')
ax2.set_ylabel('Direction 2')
ax2.set_zlabel('Energy')
ax2.set_title('Energy Landscape\n(Top-Down View)')
ax2.view_init(elev=80, azim=0)

# View 3: Side view (along transformation axis)
ax3 = fig.add_subplot(133, projection='3d')
ax3.plot_surface(xx, yy, energy, cmap='viridis', alpha=0.9, edgecolor='none')
ax3.set_xlabel('Direction 1')
ax3.set_ylabel('Direction 2')
ax3.set_zlabel('Energy')
ax3.set_title('Energy Landscape\n(Side View)')
ax3.view_init(elev=10, azim=0)

plt.tight_layout()
plt.show()

print("\nFeatures visible:")
print("  - Two valleys (wells) at approximate ligand positions")
print("  - Central barrier between the wells")
print("  - Saddle point structure along the main axis")

## 4. Contour Maps and Level Sets <a name="4-contour-maps"></a>

Contour maps provide a 2D view where:
- **Closely spaced contours** = steep energy gradient (difficult region)
- **Widely spaced contours** = flat energy (easy region)
- **Closed loops** = local minima or maxima

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Contour plot
ax1 = axes[0]
cs = ax1.contour(xx, yy, energy, levels=15, cmap='viridis')
ax1.clabel(cs, inline=True, fontsize=8)
ax1.set_xlabel('Direction 1 (A→B)')
ax1.set_ylabel('Direction 2 (⊥)')
ax1.set_title('Contour Map')
ax1.set_aspect('equal')

# Filled contour
ax2 = axes[1]
cf = ax2.contourf(xx, yy, energy, levels=20, cmap='viridis')
plt.colorbar(cf, ax=ax2, label='Energy')
ax2.set_xlabel('Direction 1 (A→B)')
ax2.set_ylabel('Direction 2 (⊥)')
ax2.set_title('Filled Contour Map')
ax2.set_aspect('equal')

# Gradient magnitude (shows difficulty)
ax3 = axes[2]
grad_y, grad_x = np.gradient(energy)
grad_mag = np.sqrt(grad_x**2 + grad_y**2)
im = ax3.imshow(grad_mag.T, origin='lower', extent=[xx.min(), xx.max(), yy.min(), yy.max()],
                cmap='hot', aspect='equal')
plt.colorbar(im, ax=ax3, label='|∇E|')
ax3.set_xlabel('Direction 1 (A→B)')
ax3.set_ylabel('Direction 2 (⊥)')
ax3.set_title('Gradient Magnitude\n(High = Difficult Regions)')

plt.tight_layout()
plt.show()

print("\nGradient magnitude interpretation:")
print("  - Bright regions (high gradient) = rapid energy change = difficult")
print("  - Dark regions (low gradient) = flat energy = easy")

## 5. Interpreting Landscape Features <a name="5-interpreting-features"></a>

### Key Features to Look For

1. **Well Depth**: How stable is each molecular state?
2. **Barrier Height**: How difficult is the transformation?
3. **Barrier Width**: How much sampling is needed?
4. **Asymmetry**: Is A→B different from B→A?

### Quantifying Features

In [ ]:
def analyze_landscape_features(energy, xx, yy):
    """
    Extract and quantify key features from the energy landscape.
    
    Returns:
        dict with feature values
    """
    features = {}
    
    # Global statistics
    features['energy_range'] = energy.max() - energy.min()
    features['energy_mean'] = energy.mean()
    features['energy_std'] = energy.std()
    
    # Find minima (potential well locations)
    from scipy.ndimage import minimum_filter
    local_min = (energy == minimum_filter(energy, size=5))
    min_indices = np.where(local_min)
    
    if len(min_indices[0]) >= 2:
        # Sort by energy value
        min_energies = energy[min_indices]
        sorted_idx = np.argsort(min_energies)
        
        # Two deepest wells
        well_1_idx = (min_indices[0][sorted_idx[0]], min_indices[1][sorted_idx[0]])
        well_2_idx = (min_indices[0][sorted_idx[1]], min_indices[1][sorted_idx[1]])
        
        features['well_1_energy'] = energy[well_1_idx]
        features['well_2_energy'] = energy[well_2_idx]
        features['well_1_position'] = (xx[well_1_idx], yy[well_1_idx])
        features['well_2_position'] = (xx[well_2_idx], yy[well_2_idx])
    
    # Barrier height (max along path between wells)
    center_slice = energy[energy.shape[0]//2, :]
    features['barrier_height'] = center_slice.max()
    features['barrier_position'] = yy[0, center_slice.argmax()]
    
    # Gradient statistics
    grad_y, grad_x = np.gradient(energy)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    features['max_gradient'] = grad_mag.max()
    features['mean_gradient'] = grad_mag.mean()
    
    return features


# Analyze our landscape
features = analyze_landscape_features(energy, xx, yy)

print("Landscape Feature Analysis")
print("=" * 40)
print(f"Energy range: {features['energy_range']:.4f}")
print(f"Mean energy: {features['energy_mean']:.4f}")
print(f"Energy std: {features['energy_std']:.4f}")
print(f"\nBarrier height: {features['barrier_height']:.4f}")
print(f"Max gradient: {features['max_gradient']:.4f}")
print(f"Mean gradient: {features['mean_gradient']:.4f}")

if 'well_1_energy' in features:
    print(f"\nWell 1 energy: {features['well_1_energy']:.4f}")
    print(f"Well 2 energy: {features['well_2_energy']:.4f}")

## 6. Interactive Visualization with Plotly <a name="6-interactive-visualization"></a>

Plotly provides interactive 3D visualizations that can be rotated, zoomed, and explored.

In [ ]:
if PLOTLY_AVAILABLE:
    # Create interactive figure
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'surface'}, {'type': 'contour'}]],
        subplot_titles=('3D Energy Surface', 'Contour Map'),
        column_widths=[0.55, 0.45]
    )
    
    # 3D Surface
    fig.add_trace(go.Surface(
        x=xx, y=yy, z=energy,
        colorscale='Viridis',
        opacity=0.9,
        showscale=True,
        colorbar=dict(title='Energy', x=0.45, len=0.8)
    ), row=1, col=1)
    
    # Add markers for wells
    if 'well_1_position' in features:
        fig.add_trace(go.Scatter3d(
            x=[features['well_1_position'][0], features['well_2_position'][0]],
            y=[features['well_1_position'][1], features['well_2_position'][1]],
            z=[features['well_1_energy'] + 0.05, features['well_2_energy'] + 0.05],
            mode='markers+text',
            marker=dict(size=8, color=['green', 'blue']),
            text=['Well 1', 'Well 2'],
            textposition='top center',
            name='Energy Wells'
        ), row=1, col=1)
    
    # Contour
    fig.add_trace(go.Contour(
        x=xx[:, 0], y=yy[0, :], z=energy.T,
        colorscale='Viridis',
        showscale=False,
        contours=dict(showlabels=True)
    ), row=1, col=2)
    
    fig.update_layout(
        title=dict(text='<b>Interactive Energy Landscape Visualization</b>', x=0.5),
        height=600,
        template='plotly_white',
        showlegend=True,
        legend=dict(x=1.05, y=0.5)
    )
    
    fig.show()
else:
    print("Plotly not available. Install with: pip install plotly")
    print("Using matplotlib visualizations above instead.")

### Adding the Geodesic Path

In [ ]:
def add_geodesic_to_landscape(fig, geodesic_path, energy_grid, xx, yy, v1, v2, center):
    """
    Add geodesic path to the landscape visualization.
    """
    from scipy.interpolate import RegularGridInterpolator
    
    # Project path to 2D
    path_2d_x = np.dot(geodesic_path - center, v1)
    path_2d_y = np.dot(geodesic_path - center, v2)
    
    # Interpolate energy along path
    interp = RegularGridInterpolator(
        (xx[:, 0], yy[0, :]), energy_grid, 
        method='linear', bounds_error=False, fill_value=0
    )
    path_energy = interp(np.column_stack([path_2d_x, path_2d_y]))
    
    # Add to 3D surface
    fig.add_trace(go.Scatter3d(
        x=path_2d_x, y=path_2d_y, z=path_energy + 0.03,
        mode='lines+markers',
        line=dict(color='red', width=6),
        marker=dict(size=3, color='red'),
        name='Geodesic Path'
    ), row=1, col=1)
    
    # Add endpoints
    fig.add_trace(go.Scatter3d(
        x=[path_2d_x[0], path_2d_x[-1]],
        y=[path_2d_y[0], path_2d_y[-1]],
        z=[path_energy[0] + 0.05, path_energy[-1] + 0.05],
        mode='markers+text',
        marker=dict(size=10, color=['green', 'blue'], symbol='diamond'),
        text=['Start (A)', 'End (B)'],
        textposition='top center',
        name='Endpoints'
    ), row=1, col=1)
    
    # Add to contour
    fig.add_trace(go.Scatter(
        x=path_2d_x, y=path_2d_y,
        mode='lines+markers',
        line=dict(color='red', width=3),
        marker=dict(size=4),
        name='Geodesic',
        showlegend=False
    ), row=1, col=2)
    
    return fig, path_energy

print("Function defined. Use with geodesic path to visualize on landscape.")

## Summary

### Visualization Techniques

| Technique | Best For | Key Insight |
|-----------|----------|-------------|
| 3D Surface | Overall shape | Valleys, barriers, saddles |
| Contour Map | Quantitative analysis | Energy level spacing |
| Gradient Magnitude | Identifying difficult regions | High gradient = hard |
| Interactive (Plotly) | Exploration | Rotate, zoom, inspect |

### Key Takeaways

1. The **energy landscape** encodes transformation difficulty spatially

2. **Valleys** represent stable molecular states (easy to sample)

3. **Barriers** indicate transformation difficulty (need more λ-windows)

4. The **geodesic** follows the minimum energy path through the landscape

5. **Gradient magnitude** shows where sampling will be most challenging

### Next Steps

In the final notebook, we'll put everything together in a practical workflow for analyzing new molecular pairs.